# 2장 텍스트 데이터 다루기: 따라 치기 노트북

『밑바닥부터 만들면서 배우는 LLM』 2장을 따라 치며 공부하는 노트북입니다.

**진행 방법**
1. 절마다 책을 보며 `✏️` 셀에 코드를 직접 입력하고 실행하세요. 셀은 필요한 만큼 추가하면 됩니다.
2. 출력(토큰, 토큰 ID, 텐서 shape)을 **매번 책과 비교**하세요.
3. 주석은 책 문장을 옮기지 말고 **"이 줄이 왜 필요한지"를 내 말로** 쓰세요.
4. 🔍 **확인 셀**에서 `✅`가 나오면 다음 절로 넘어가세요.
5. 장을 마치면 `notes.md`를 채우고, 핵심 두 가지(토크나이저 클래스, 슬라이딩 윈도우 데이터셋)를 보지 않고 다시 짜 보세요.

막히면 [책의 공식 코드](https://github.com/rasbt/LLMs-from-scratch/tree/main/ch02/01_main-chapter-code)를 참고하세요.

## 0. 환경 설정
Colab에는 `torch`가 설치되어 있고, GPT-2의 BPE 토크나이저인 `tiktoken`만 추가로 설치합니다. 2장은 GPU가 없어도 됩니다.

In [30]:
!pip install -q tiktoken

from importlib.metadata import version #버전 정보 조회 함수
print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.14.1+cpu
tiktoken version: 0.14.0


## 데이터 불러오기
같은 폴더의 `the-verdict.txt`를 읽습니다. Colab처럼 파일이 없으면 이 저장소에서 내려받습니다.

In [31]:
import os
import urllib.request

file_path = "the-verdict.txt"
if not os.path.exists(file_path): #path에 없으면
    url = "https://raw.githubusercontent.com/uisang2004-maker/LLM-/main/ch02_tokenizer/the-verdict.txt"
    urllib.request.urlretrieve(url, file_path) #url로 가져오기

with open(file_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

print("총 문자 수:", len(raw_text))
print(raw_text[:99])

총 문자 수: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


## 2.1 단어 임베딩 이해하기
코드 없이 개념을 읽는 절입니다. 읽고 나서 **"임베딩이란?"** 을 한 문장으로 `notes.md`에 적어 보세요.

## 2.2 텍스트 토큰화하기
정규 표현식(`re`)으로 텍스트를 단어와 구두점으로 나눕니다. 나눈 결과를 `preprocessed`에 담으세요.

In [32]:
# ✏️ 책을 보며 직접 입력하세요: re.split으로 텍스트 나누기 → preprocessed
import re
result=re.split(r'([,.:;?_!"()\']|--|\s)',raw_text)  # 공백·구두점·'--' 기준으로 나누고, 괄호(캡처 그룹) 덕분에 구두점도 토큰으로 남김
preprocessed=[item.strip() for item in result if item.strip()]

In [33]:
# 🔍 확인
print("토큰 수:", len(preprocessed))
print(preprocessed[:30])
print("✅ 책의 토큰 수, 앞부분 출력과 같은지 비교하세요")

토큰 수: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
✅ 책의 토큰 수, 앞부분 출력과 같은지 비교하세요


## 2.3 토큰을 토큰 ID로 변환하기
고유한 토큰에 번호를 매겨 어휘 사전(`vocab`)을 만들고, `encode`와 `decode`가 있는 `SimpleTokenizerV1` 클래스를 만듭니다.

In [34]:
# ✏️ 책을 보며 직접 입력하세요: 어휘 사전 vocab 만들기
vocab= {token:integer for integer,token in enumerate(sorted(set(preprocessed)))}  # 중복을 없애고 정렬한 토큰마다 번호를 붙임 (정렬해야 실행할 때마다 번호가 같음)


In [35]:
# ✏️ 책을 보며 직접 입력하세요: SimpleTokenizerV1 클래스 (encode, decode)
class SimpleTokenizerV1:
    def __init__(self,vocab):
      self.str_to_int=vocab
      self.int_to_str={i:s for s,i in vocab.items()}
    def encode(self,text):
      preprocessed=re.split(r'([,.:;?_!"()\']|--|\s)',text)  # 텍스트 전처리 (2.2절과 같은 규칙)
      preprocessed=[item.strip() for item in preprocessed if item.strip()]#공백제거
      ids=[self.str_to_int[s] for s in preprocessed]#단어들을 사전에 있는 id로 변환
      return ids
    def decode(self,ids):
      text=" ".join([self.int_to_str[i] for i in ids])
      text=re.sub(r'\s+([,.?!"()\'])',r'\1',text)# 구두점(,.?!"()') 앞의 공백 지우기
      return text





In [36]:
# 🔍 확인: 문장 → ID → 문장
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print("어휘 사전 크기:", len(vocab))
print(ids)
print(tokenizer.decode(ids))
assert isinstance(ids, list) and all(isinstance(i, int) for i in ids), "encode는 정수 리스트를 돌려줘야 해요"
print("✅ encode / decode 동작 확인")

어휘 사전 크기: 1130
[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.
✅ encode / decode 동작 확인


## 2.4 특수 문맥 토큰 추가하기
어휘 사전에 없는 단어를 위한 `<|unk|>`와 문서 경계를 표시하는 `<|endoftext|>`를 추가하고, `SimpleTokenizerV2`를 만듭니다.

In [37]:
# ✏️ 책을 보며 직접 입력하세요: 특수 토큰을 추가한 어휘 사전
all_tokens=sorted(list(set(preprocessed)))
all_tokens.extend(["<|unk|>","<|endoftext|>"])
vocab={token:integer for integer,token in enumerate(all_tokens)}


In [38]:
# ✏️ 책을 보며 직접 입력하세요: SimpleTokenizerV2 클래스
class SimpleTokenizerV2:
  def __init__(self,vocab):
    self.str_to_int=vocab
    self.int_to_str={i:s for s,i in vocab.items()}
  def encode(self,text):
    preprocessed=re.split(r'([,.:;?_!"()\']|--|\s)',text)
    preprocessed=[item.strip() for item in preprocessed if item.strip()]
    preprocessed=[item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
    ids=[self.str_to_int[s] for s in preprocessed]
    return ids
  def decode(self,ids):
    text=" ".join([self.int_to_str[i] for i in ids])
    text=re.sub(r'\s+([,.?!"()\'])',r'\1',text)
    return text



In [39]:
# 🔍 확인: 사전에 없는 단어가 <|unk|>로 바뀌는지
tokenizer = SimpleTokenizerV2(vocab)
text = " <|endoftext|> ".join(["Hello, do you like tea?", "In the sunlit terraces of the palace."])
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))
assert "<|unk|>" in tokenizer.decode(tokenizer.encode(text)), "사전에 없는 'Hello' 등이 <|unk|>로 바뀌어야 해요"
print("✅ 특수 토큰 처리 확인")

[1130, 5, 355, 1126, 628, 975, 10, 1131, 55, 988, 956, 984, 722, 988, 1130, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.
✅ 특수 토큰 처리 확인


## 2.5 바이트 페어 인코딩 (BPE)
GPT-2가 쓰는 BPE 토크나이저를 `tiktoken`으로 불러옵니다. 공부 노트의 BPE 정리와 비교해 보세요. 사전에 없는 단어도 `<|unk|>` 없이 조각으로 나뉘는 게 핵심이에요.

In [40]:
# ✏️ 책을 보며 직접 입력하세요: tiktoken으로 gpt2 토크나이저 불러와서 encode / decode
!pip install tiktoken
import tiktoken
tokenizer=tiktoken.get_encoding("gpt2")

In [41]:
# 🔍 확인: 처음 보는 단어도 조각으로 나뉘는지
import tiktoken
_bpe = tiktoken.get_encoding("gpt2")
for word in ["Akwirw ier", "someunknownPlace"]:
    _ids = _bpe.encode(word)
    print(word, "→", _ids, "→", [_bpe.decode([i]) for i in _ids])
print("어휘 사전 크기:", _bpe.n_vocab)
print("✅ BPE는 <|unk|> 없이 어떤 문자열도 표현할 수 있어요")

Akwirw ier → [33901, 86, 343, 86, 220, 959] → ['Ak', 'w', 'ir', 'w', ' ', 'ier']
someunknownPlace → [11246, 34680, 27271] → ['some', 'unknown', 'Place']
어휘 사전 크기: 50257
✅ BPE는 <|unk|> 없이 어떤 문자열도 표현할 수 있어요


## 2.6 슬라이딩 윈도우로 데이터 샘플링하기 ⭐
"다음 토큰 맞히기"를 학습 데이터로 바꾸는 핵심 절이에요. **입력을 한 칸 밀면 정답**이 됩니다. `GPTDatasetV1`과 `create_dataloader_v1`을 만드세요. 문서 분류와 NER에서 만든 `Dataset`과 비교해 보세요.

In [42]:
# ✏️ 책을 보며 직접 입력하세요: 입력과 한 칸 밀린 정답(target) 예시 출력
with open("the-verdict.txt","r",encoding="utf-8") as f:
  raw_text=f.read()
enc_text=tokenizer.encode(raw_text)
enc_sample=enc_text[50:]
print(len(enc_text))
context_size=4
for i in range(1, context_size+1):
  context=enc_sample[:i]#i전까지의 값(입력값)
  desired=enc_sample[i]#i값(예측해야 할 값(레이블))
  print(tokenizer.decode(context),"-->",tokenizer.decode([desired]))


5145
 and -->  established
 and established -->  himself
 and established himself -->  in
 and established himself in -->  a


In [43]:
# ✏️ 책을 보며 직접 입력하세요: GPTDatasetV1 클래스=데이터셋에서 개별 행 추출
import torch
from torch.utils.data import Dataset,DataLoader
class GPTDatasetV1(Dataset):
  def __init__(self,txt,tokenizer,max_length,stride):
    self.input_ids=[]  # 입력 조각들
    self.target_ids=[]  # 정답 조각들 (입력을 한 칸 민 다음 토큰)
    token_ids= tokenizer.encode(txt)#입력받은 txt를 토큰화
    for i in range(0, len(token_ids)-max_length,stride):#len(token_ids) - max_length 전까지,stride 간격으로 윈도우 옮김
        input_chunk=token_ids[i:i+max_length] # 입력 조각: i부터 max_length개
        target_chunk=token_ids[i+1:i+max_length+1] # 정답 조각: 입력을 한 칸 뒤로 민 것
        self.input_ids.append(torch.tensor(input_chunk))  #input 청크 넣기
        self.target_ids.append(torch.tensor(target_chunk))
  def __len__(self):# 전체 길이
    return len(self.input_ids)
  def __getitem__(self,idx):#특정 인덱스의 input과 타겟 ids 반환
    return self.input_ids[idx],self.target_ids[idx]


In [44]:
# ✏️ 책을 보며 직접 입력하세요: create_dataloader_v1 함수
def create_dataloader_v1(txt,batch_size=4,max_length=256,stride=128,shuffle=True,drop_last=True,num_workers=0):
  tokenizer=tiktoken.get_encoding("gpt2") #토크나이저 초기화
  dataset=GPTDatasetV1(txt,tokenizer,max_length,stride)
  dataloader=DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle=shuffle,
      drop_last=drop_last,#배치 사이즈보다 작을경우 마지막 배치 삭제
      num_workers=num_workers#전처리에 사용할 cpu프로세서 갯수
  )
  return dataloader

In [45]:
# 🔍 확인: 정답이 입력을 한 칸 민 것인지
import torch
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
inputs, targets = next(iter(dataloader))
print("inputs:\n", inputs)
print("targets:\n", targets)
assert inputs.shape == (8, 4) and targets.shape == (8, 4), f"shape이 (8, 4)여야 해요: {tuple(inputs.shape)}"
assert torch.equal(inputs[:, 1:], targets[:, :-1]), "정답은 입력을 한 칸 왼쪽으로 민 것이어야 해요"
print("✅ 슬라이딩 윈도우 데이터셋 확인")

inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
✅ 슬라이딩 윈도우 데이터셋 확인


## 2.7 토큰 임베딩 만들기
`torch.nn.Embedding`으로 토큰 ID를 벡터로 바꿉니다. 임베딩 층은 **번호로 행을 찾아오는 표**라는 점을 확인해 보세요.

In [46]:
# ✏️ 책을 보며 직접 입력하세요: 작은 Embedding 층 만들어서 가중치와 결과 비교
input_ids=torch.tensor([2,3,5,1])#4개의 입력 토큰 ids
vocab_size=6 #어휘 사전의 크기
output_dim=3 #출력 텐서의 차원 수
torch.manual_seed(123) #랜덤시드 고정
embedding_layer= torch.nn.Embedding(vocab_size,output_dim) #어휘사전 크기+출력 차원수를 입력하여 임베딩 층 초기화
print(embedding_layer.weight) #임베딩 층의 가중치들을 출력 = 작고 랜덤한 값들=> 이후 LLM훈련 과정에서 최적화
#=> 어휘 사전 6개의 단어들에 각각 3 임베딩 차원만큼의 가중치가 할당
print(embedding_layer(input_ids)) #임베딩 가중치에서 룩업 연산을 통해 구해짐(가중치 행렬의 해당 인덱스 행을 추출해 그 벡터가 된다)



Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


## 2.8 단어 위치 인코딩하기
토큰 임베딩에 **위치 임베딩**을 더해 LLM의 입력을 완성합니다. 공부 노트의 "임베딩 + 위치 정보"가 바로 이것이에요.

In [49]:
# ✏️ 책을 보며 직접 입력하세요: 토큰 임베딩 + 위치 임베딩 → input_embeddings
token_embedding_layer=torch.nn.Embedding(50257,256) #50257 어휘사전 크기를 가진 토크나이저에서 각 배치의 토큰 id를 256 차원 벡터로 엠베딩
max_length=4  # 한 샘플(입력 조각)의 토큰 수
dataloader=create_dataloader_v1(raw_text,batch_size=8,max_length=max_length,stride=max_length,shuffle=False) #배치 크기=8 => 토큰 id 텐서 차원= 8x4
data_iter=iter(dataloader) #이터레이터로 한 배치씩 꺼냄
inputs,targets=next(data_iter)
print(inputs)
print(inputs.shape)
token_embeddings=token_embedding_layer(inputs) #input토큰 임베딩
pos_embedding_layer=torch.nn.Embedding(max_length,256) # 위치 임베딩 층: 위치 0~max_length-1마다 토큰 임베딩과 같은 256차원 벡터
pos_embeddings=pos_embedding_layer(torch.arange(max_length))
input_embeddings=token_embeddings+pos_embeddings #위치 임베딩 더하기
print(input_embeddings.shape)#임베딩 완료된 토큰 ids의 shape= 8x4x256차원의 텐서


tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
torch.Size([8, 4])
torch.Size([8, 4, 256])


In [50]:
# 🔍 확인: 최종 입력의 shape
print(input_embeddings.shape)
assert tuple(input_embeddings.shape) == (8, 4, 256), "책 설정(batch 8, max_length 4, 차원 256)이면 (8, 4, 256)이어야 해요"
print("✅ (배치, 토큰 수, 임베딩 차원) = (8, 4, 256)")

torch.Size([8, 4, 256])
✅ (배치, 토큰 수, 임베딩 차원) = (8, 4, 256)
